In [ ]:
import numpy as np
import scipy.io as sio
from scipy.signal import butter, sosfiltfilt
from scipy.signal import welch
from sklearn.feature_selection import mutual_info_regression
from joblib import Parallel, delayed
from pathlib import Path

#load dataset

data = sio.loadmat("./data/DREAMER.mat")
mat = sio.loadmat(
    "data/DREAMER.mat",
    squeeze_me = True,
    struct_as_record=False
)


dreamer = mat["DREAMER"]

In [48]:
#dataset overview
print("Number of subjects:", dreamer.noOfSubjects)
print("Number of videos:", dreamer.noOfVideoSequences)
print("EEG sampling rate:", dreamer.EEG_SamplingRate)
print("ECG sampling rate:", dreamer.ECG_SamplingRate)

print("\nEEG electrodes:")
print(dreamer.EEG_Electrodes)

Number of subjects: 23
Number of videos: 18
EEG sampling rate: 128
ECG sampling rate: 256

EEG electrodes:
['AF3' 'F7' 'F3' 'FC5' 'T7' 'P7' 'O1' 'O2' 'P8' 'T8' 'FC6' 'F4' 'F8' 'AF4']


In [49]:
#extract EEG 
all_eeg = []

for s in range(dreamer.noOfSubjects):
    subject = dreamer.Data[s]

    #EEG structure
    eeg_struct = subject.EEG

    stimuli = eeg_struct.stimuli

    subject_eeg = []

    for video in range(dreamer.noOfVideoSequences):
        eeg = np.asarray(stimuli[video])
        subject_eeg.append(eeg)

    all_eeg.append(subject_eeg)


In [50]:
# Convert to NumPy object array
all_eeg = np.array(all_eeg , dtype=object)
print("Array shape:", all_eeg.shape)
print("Array dtype:", all_eeg.dtype)

Array shape: (23, 18)
Array dtype: object


In [51]:
#checking first record
first = all_eeg[0,0]
print("\nFirst recording:")
print("Subject: 1")
print("Video: 1")
print("Shape:", first.shape)

print("\nFirst 5 samples:")
print(first[:5])


First recording:
Subject: 1
Video: 1
Shape: (25472, 14)

First 5 samples:
[[4388.20512821 4102.56410256 4219.48717949 4465.12820513 4370.76923077
  4399.48717949 4443.07692308 4023.07692308 4365.12820513 4310.25641026
  3953.84615385 4454.35897436 4326.15384615 4165.12820513]
 [4375.8974359  4093.84615385 4252.82051282 4522.56410256 4435.8974359
  4411.79487179 4488.71794872 4108.71794872 4399.48717949 4384.61538462
  4007.69230769 4466.66666667 4372.82051282 4247.17948718]
 [4378.46153846 4091.28205128 4230.25641026 4488.20512821 4370.25641026
  4402.56410256 4461.02564103 4077.43589744 4378.46153846 4328.71794872
  3986.15384615 4461.02564103 4328.20512821 4203.58974359]
 [4393.84615385 4101.02564103 4193.33333333 4418.97435897 4270.25641026
  4392.30769231 4411.28205128 3982.56410256 4336.41025641 4213.33333333
  3930.25641026 4442.56410256 4261.02564103 4100.        ]
 [4396.41025641 4108.71794872 4210.76923077 4436.41025641 4310.76923077
  4401.02564103 4426.66666667 3980.5128205

In [52]:
# ============================================================
# Prepare DREAMER EEG + V/A/D
# ============================================================
import pandas as pd

electrodes = [
    "AF3", "F7", "F3", "FC5",
    "T7", "P7", "O1", "O2",
    "P8", "T8", "FC6", "F4",
    "F8", "AF4"
]

dataset = []

for s in range(dreamer.noOfSubjects):

    subject = dreamer.Data[s]

    for v in range(dreamer.noOfVideoSequences):

        eeg = np.asarray(
            subject.EEG.stimuli[v],
            dtype=np.float32
        )

        eeg_df = pd.DataFrame(
            eeg,
            columns=electrodes
        )

        # Emotion labels
        valence = float(subject.ScoreValence[v])
        arousal = float(subject.ScoreArousal[v])
        dominance = float(subject.ScoreDominance[v])

        # Save trial
        dataset.append({
            "subject": s + 1,
            "trial": v + 1,
            "eeg": eeg_df,
            "valence": valence,
            "arousal": arousal,
            "dominance": dominance
        })


print("Total trials:", len(dataset))

Total trials: 414


In [53]:
#show and check dataset
print("Number of trials:", len(dataset))

print("\nFirst trial:")
print("Subject:", dataset[0]["subject"])
print("Trial:", dataset[0]["trial"])

print("\nV/A/D:")
print("Valence:", dataset[0]["valence"])
print("Arousal:", dataset[0]["arousal"])
print("Dominance:", dataset[0]["dominance"])

print("\nEEG shape:")
print(dataset[0]["eeg"].shape)

display(dataset[0]["eeg"].head())

Number of trials: 414

First trial:
Subject: 1
Trial: 1

V/A/D:
Valence: 4.0
Arousal: 3.0
Dominance: 2.0

EEG shape:
(25472, 14)


,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4
0,4388.205078,4102.563965,4219.487305,4465.128418,4370.769043,4399.487305,4443.077148,4023.076904,4365.128418,4310.256348,3953.846191,4454.358887,4326.153809,4165.128418
1,4375.897461,4093.846191,4252.820312,4522.563965,4435.897461,4411.794922,4488.717773,4108.717773,4399.487305,4384.615234,4007.692383,4466.666504,4372.820312,4247.179688
2,4378.461426,4091.281982,4230.256348,4488.205078,4370.256348,4402.563965,4461.025879,4077.435791,4378.461426,4328.717773,3986.153809,4461.025879,4328.205078,4203.589844
3,4393.846191,4101.025879,4193.333496,4418.974121,4270.256348,4392.307617,4411.282227,3982.564209,4336.410156,4213.333496,3930.256348,4442.563965,4261.025879,4100.000000
4,4396.410156,4108.717773,4210.769043,4436.410156,4310.769043,4401.025879,4426.666504,3980.512939,4349.743652,4238.461426,3945.128174,4446.666504,4289.743652,4115.384766


In [ ]:
# Create windows for all DREAMER trials

window_seconds = 2
overlap = 0.5

fs = int(dreamer.EEG_SamplingRate)

window_size = int(window_seconds * fs)
step_size = int(window_size * (1 - overlap))

all_windows = []
all_valence = []
all_arousal = []
all_dominance = []

all_subjects = []
all_trials = []

# Process every trial
for trial_data in dataset:

    eeg_data = trial_data["eeg"].values

    valence = trial_data["valence"]
    arousal = trial_data["arousal"]
    dominance = trial_data["dominance"]

    subject_id = trial_data["subject"]
    trial_id = trial_data["trial"]

    # Create windows for this trial
    for start in range(
        0,
        len(eeg_data) - window_size + 1,
        step_size
    ):

        end = start + window_size

        window_data = eeg_data[start:end]

        all_windows.append(window_data)

        # Same label for every window of this trial
        all_valence.append(valence)
        all_arousal.append(arousal)
        all_dominance.append(dominance)

        all_subjects.append(subject_id)
        all_trials.append(trial_id)


# Convert to NumPy arrays
all_windows = np.asarray(
    all_windows,
    dtype=np.float32
)

all_valence = np.asarray(
    all_valence,
    dtype=np.float32
)

all_arousal = np.asarray(
    all_arousal,
    dtype=np.float32
)

all_dominance = np.asarray(
    all_dominance,
    dtype=np.float32
)

all_subjects = np.asarray(all_subjects)
all_trials = np.asarray(all_trials)


print("Window size:", window_size)
print("Step size:", step_size)

print("All windows shape:", all_windows.shape)

print("Valence shape:", all_valence.shape)
print("Arousal shape:", all_arousal.shape)
print("Dominance shape:", all_dominance.shape)

print("Subjects shape:", all_subjects.shape)
print("Trials shape:", all_trials.shape)

Window size: 256
Step size: 128
All windows shape: (85330, 256, 14)
Valence shape: (85330,)
Arousal shape: (85330,)
Dominance shape: (85330,)
Subjects shape: (85330,)
Trials shape: (85330,)


In [ ]:
#filtering

# Sampling rate
fs = int(dreamer.EEG_SamplingRate)

# Filter settings
lowcut = 0.5
highcut = 45.0
filter_order = 4

# Band-pass filter
bpf = butter(
    filter_order,
    [lowcut, highcut],
    btype="bandpass",
    fs=fs,
    output="sos"
)

# Filtering all windows
filtered_windows = sosfiltfilt(
    bpf,
    all_windows,
    axis=1
)

filtered_windows = filtered_windows.astype(np.float32)

print("Original shape:", all_windows.shape)
print("Filtered shape:", filtered_windows.shape)
print("Data type:", filtered_windows.dtype)

Original shape: (85330, 256, 14)
Filtered shape: (85330, 256, 14)
Data type: float32


In [ ]:
#correlation

# number of windows
n_windows = filtered_windows.shape[0]

# calculate correlation for all windows
correlation_matrices = np.zeros(
    (n_windows, 14, 14),
    dtype=np.float32
)

for i in range(n_windows):
    correlation_matrices[i] = np.corrcoef(
        filtered_windows[i],
        rowvar=False
    )

print("Correlation matrices shape:", correlation_matrices.shape)

Correlation matrices shape: (85330, 14, 14)


In [57]:
# test => window = filtered_windows[0]

# Sampling rate
fs = int(dreamer.EEG_SamplingRate)

print("Window shape:", window.shape)
print("Sampling rate:", fs)

Window shape: (256, 14)
Sampling rate: 128


In [ ]:
# FFT in Window 
fft_values = np.fft.rfft(window, axis=0)


frequencies = np.fft.rfftfreq(
    window.shape[0],
    d=1 / fs
)

#check date
print("FFT shape:", fft_values.shape)
print("Frequency shape:", frequencies.shape)

print("\nFirst frequencies:")
print(frequencies[:15])

FFT shape: (129, 14)
Frequency shape: (129,)

First frequencies:
[0.  0.5 1.  1.5 2.  2.5 3.  3.5 4.  4.5 5.  5.5 6.  6.5 7. ]


In [ ]:
# calculate Power Spectrum (PSD) 
power_spectrum = np.abs(fft_values) ** 2

print("Power spectrum shape:", power_spectrum.shape)

Power spectrum shape: (129, 14)

AF3 power values:
[   7752.8633  705233.2    3904563.      186967.17    183821.61
  316082.1     129486.65     82117.28     56329.406    41603.637 ]


In [ ]:
# Welch PSD for windows
freqs, psd = welch(
    window,
    fs=fs,
    axis=0,
    nperseg=256
)

print("Frequencies shape:", freqs.shape)
print("PSD shape:", psd.shape)

print("\nFirst frequencies:")
print(freqs[:15])

Frequencies shape: (129,)
PSD shape: (129, 14)

First frequencies:
[0.  0.5 1.  1.5 2.  2.5 3.  3.5 4.  4.5 5.  5.5 6.  6.5 7. ]


In [61]:
#test

# define band of frequency
alpha_band = (8, 13)
beta_band = (13, 30)

# find frequency
alpha_idx = (freqs >= alpha_band[0]) & (freqs < alpha_band[1])
beta_idx = (freqs >= beta_band[0]) & (freqs <= beta_band[1])



print("Alpha frequencies:")
print(freqs[alpha_idx])

print("\nBeta frequencies:")
print(freqs[beta_idx])

Alpha frequencies:
[ 8.   8.5  9.   9.5 10.  10.5 11.  11.5 12.  12.5]

Beta frequencies:
[13.  13.5 14.  14.5 15.  15.5 16.  16.5 17.  17.5 18.  18.5 19.  19.5
 20.  20.5 21.  21.5 22.  22.5 23.  23.5 24.  24.5 25.  25.5 26.  26.5
 27.  27.5 28.  28.5 29.  29.5 30. ]


In [ ]:
#power

# number of windows
n_windows = filtered_windows.shape[0]

# output arrays
alpha_power_all = np.zeros(
    (n_windows, 14),
    dtype=np.float32
)

beta_power_all = np.zeros(
    (n_windows, 14),
    dtype=np.float32
)

# calculate power for each window
for i in range(n_windows):
    window = filtered_windows[i]

    # Welch PSD
    freqs, psd = welch(
        window,
        fs=fs,
        axis=0,
        nperseg=256
    )

    df = freqs[1] - freqs[0]

    # bands
    alpha_idx = (freqs >= 8) & (freqs < 13)
    beta_idx = (freqs >= 13) & (freqs <= 30)

    # Alpha Power
    alpha_power_all[i] = np.sum(
        psd[alpha_idx, :] * df,
        axis=0
    )

    # Beta Power
    beta_power_all[i] = np.sum(
        psd[beta_idx, :] * df,
        axis=0
    )


print("Alpha Power shape:", alpha_power_all.shape)
print("Beta Power shape:", beta_power_all.shape)

Alpha Power shape: (85330, 14)
Beta Power shape: (85330, 14)


In [63]:
# EEG electrode names
electrodes = [
    "AF3", "F7", "F3", "FC5", "T7", "P7", "O1",
    "O2", "P8", "T8", "FC6", "F4", "F8", "AF4"
]

# Left-Right electrode pairs
pairs = [
    ("AF3", "AF4"),
    ("F7", "F8"),
    ("F3", "F4"),
    ("FC5", "FC6"),
    ("T7", "T8"),
    ("P7", "P8"),
    ("O1", "O2")
]

# Convert electrode names to indices
pair_indices = [
    (electrodes.index(left), electrodes.index(right))
    for left, right in pairs
]

for (left, right), (li, ri) in zip(pairs, pair_indices):
    print(f"{left} ({li}) <-> {right} ({ri})")

AF3 (0) <-> AF4 (13)
F7 (1) <-> F8 (12)
F3 (2) <-> F4 (11)
FC5 (3) <-> FC6 (10)
T7 (4) <-> T8 (9)
P7 (5) <-> P8 (8)
O1 (6) <-> O2 (7)


In [64]:
# Left-Right electrode pairs for alpha
pair_indices = [
    (0, 13),  # AF3 - AF4
    (1, 12),  # F7  - F8
    (2, 11),  # F3  - F4
    (3, 10),  # FC5 - FC6
    (4, 9),   # T7  - T8
    (5, 8),   # P7  - P8
    (6, 7)    # O1  - O2
]

# Small value for noise
epsilon = 1e-10


# number of windows × number of left-right pairs
alpha_asymmetry = np.zeros(
    (alpha_power_all.shape[0], len(pair_indices)),
    dtype=np.float32
)

# Calculate Alpha Asymmetry
for i, (left, right) in enumerate(pair_indices):

    left_power = alpha_power_all[:, left]
    right_power = alpha_power_all[:, right]

    alpha_asymmetry[:, i] = np.log(
        (left_power + epsilon) /
        (right_power + epsilon)
    )

print("Alpha Asymmetry shape:", alpha_asymmetry.shape)

print("\nFirst 5 windows:")
print(alpha_asymmetry[:5])

Alpha Asymmetry shape: (85330, 7)

First 5 windows:
[[-6.2068605  -3.8554692   0.99643886  0.15480837  0.02878639 -2.2205343
  -0.21694717]
 [-4.514204   -3.7945163   1.1289802   0.4349583  -0.0066219  -1.9997221
  -0.331589  ]
 [-5.9470863  -3.9124982   0.83053476  0.3283982   0.03359229 -2.1115875
  -0.3971667 ]
 [-4.752528   -3.8249621   0.7952691   0.18713379  0.08951646 -2.271279
  -0.25320157]
 [-5.3899255  -3.8319416   0.7860191   0.09923737 -0.08016727 -2.4677088
  -0.3069448 ]]


In [67]:
# number windows × number left-right pairs for beta 
beta_asymmetry = np.zeros(
    (beta_power_all.shape[0], len(pair_indices)),
    dtype=np.float32
)

# Calculate Beta Asymmetry
for i, (left, right) in enumerate(pair_indices):

    left_power = beta_power_all[:, left]
    right_power = beta_power_all[:, right]

    beta_asymmetry[:, i] = np.log(
        (left_power + epsilon) /
        (right_power + epsilon)
    )

print("Beta Asymmetry shape:", beta_asymmetry.shape)

print("\nFirst 5 windows:")
print(beta_asymmetry[:5])

Beta Asymmetry shape: (85330, 7)

First 5 windows:
[[-3.9556289  -4.5774364   2.1397183   0.5734785  -0.3087768  -2.3548086
  -1.0967517 ]
 [-3.9852777  -4.225094    1.84531     0.33173385 -0.3094326  -2.4700549
  -1.5945317 ]
 [-5.411494   -5.5471745   1.3739616   1.0909739  -0.1698515  -3.771644
  -1.2858313 ]
 [-4.5353866  -4.416244    1.6217272  -0.04803373 -0.36272854 -2.4527824
  -2.1787083 ]
 [-3.8153214  -4.0335736   2.0103433   0.35741046 -0.2838634  -2.323538
  -1.5424984 ]]


In [ ]:
# Mutual Information (MI)

def calculate_mi_window(window):
    n_channels = window.shape[1]

    mi_matrix = np.zeros(
        (n_channels, n_channels),
        dtype=np.float32
    )

    for i in range(n_channels):
        for j in range(i + 1, n_channels):

            mi_value = mutual_info_regression(
                window[:, i].reshape(-1, 1),
                window[:, j],
                random_state=42
            )[0]

            mi_matrix[i, j] = mi_value
            mi_matrix[j, i] = mi_value

    return mi_matrix


# Calculate MI for all windows
mi_all = np.array(
    Parallel(n_jobs=-1, verbose=10)(
        delayed(calculate_mi_window)(window)
        for window in filtered_windows
    ),
    dtype=np.float32
)

print("MI calculation finished.")
print("MI shape:", mi_all.shape)

MI test shape: (100, 14, 14)


In [ ]:
# Save processed features

output_dir  = Path("processed_data")

np.save(output_dir / "correlation.npy", correlation_matrices)
np.save(output_dir / "mi.npy", mi_all)

np.save(output_dir / "alpha_power.npy", alpha_power_all)
np.save(output_dir / "beta_power.npy", beta_power_all)

np.save(output_dir / "alpha_asymmetry.npy", alpha_asymmetry)
np.save(output_dir / "beta_asymmetry.npy", beta_asymmetry)


In [73]:
# Labels for the current trial
trial_valence = float(subject.ScoreValence[v])
trial_arousal = float(subject.ScoreArousal[v])
trial_dominance = float(subject.ScoreDominance[v])

# Assign the trial labels to all windows
valence_labels = np.full(
    len(windows),
    trial_valence,
    dtype=np.float32
)

arousal_labels = np.full(
    len(windows),
    trial_arousal,
    dtype=np.float32
)

dominance_labels = np.full(
    len(windows),
    trial_dominance,
    dtype=np.float32
)

print("Number of windows:", len(windows))
print("Valence labels:", valence_labels.shape)
print("Arousal labels:", arousal_labels.shape)
print("Dominance labels:", dominance_labels.shape)

Number of windows: 185
Valence labels: (185,)
Arousal labels: (185,)
Dominance labels: (185,)
